# Rewriter dataset check for Colab

This notebook loads the private training pilot and checks its formatting and native Liquid token spans. It loads no model weights and runs no training. Add `HF_TOKEN` in Colab Secrets with read access to the private dataset. GPU training and the 68-case before/after evaluation are the next step.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==5.21.0",
        "huggingface_hub==1.33.0",
        "jinja2==3.1.6",
    ]
)

In [ ]:
import hashlib
import json
from collections import Counter
from pathlib import Path

from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

repo = "jayshah5696/finetuning-labs-rewriter-training"
token = userdata.get("HF_TOKEN")
revision = HfApi(token=token).repo_info(repo, repo_type="dataset").sha


def download(name):
    path = hf_hub_download(repo, name, repo_type="dataset", revision=revision, token=token)
    return Path(path).read_bytes()


manifest = json.loads(download("manifest.json"))
payloads = {}
for name, expected in manifest["files"].items():
    data = download(name)
    assert hashlib.sha256(data).hexdigest() == expected, name
    payloads[name] = data
quality = json.loads(payloads["quality.json"])
support = json.loads(payloads["supporting.json"])
print({"repo": repo, "immutable_revision": revision, "splits": quality["splits"]})

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "LiquidAI/LFM2.5-230M", revision="40cb2ad3b3044d5a41eee083a6103c8b523afa45"
)
splits, prepared, rows_by_split = {}, {}, {}
for split in ("train", "validation"):
    rows = [json.loads(line) for line in payloads[f"{split}.jsonl"].splitlines()]
    guides = [json.loads(line) for line in payloads[f"sft/{split}.jsonl"].splitlines()]
    meta = [item for item in support if item["split"] == split]
    assert len(rows) == len(guides) == len(meta) == quality["splits"][split]
    prepared[split], rows_by_split[split] = [], guides
    for row, guide, item in zip(rows, guides, meta, strict=True):
        assert set(row) == {"input", "output", "mode", "checks"}
        assert set(guide) == {"system", "user", "assistant"}
        assert guide["assistant"] == row["output"]
        encoded = json.dumps(
            row, ensure_ascii=False, sort_keys=True, separators=(",", ":")
        ).encode()
        assert hashlib.sha256(encoded).hexdigest() == item["row_sha256"]
        if item["length"] == "long":
            assert 1000 <= len(tokenizer.encode(row["input"], add_special_tokens=False)) <= 2000
        messages = [
            {"role": "system", "content": guide["system"]},
            {"role": "user", "content": guide["user"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=False
        )
        full = tokenizer.apply_chat_template(
            messages + [{"role": "assistant", "content": guide["assistant"]}],
            tokenize=True,
            return_dict=False,
        )
        assert full[: len(prompt)] == prompt
        assert len(full) <= 4096 and len(prompt) + 2048 <= 4096
        end = len(prompt) + full[len(prompt) :].index(tokenizer.eos_token_id) + 1
        assert len(prompt) < end - 1 and end - len(prompt) <= 2048
        assert not tokenizer.decode(full[end:]).strip()
        labels = [-100] * len(prompt) + full[len(prompt) : end] + [-100] * (len(full) - end)
        assert sum(label != -100 for label in labels) == item["tokens"]["assistant_tokens"]
        prepared[split].append({"input_ids": full, "labels": labels})
    splits[split] = dict(Counter(row["mode"] for row in rows))

train_groups = {item["source_group"] for item in support if item["split"] == "train"}
dev_groups = {item["source_group"] for item in support if item["split"] == "validation"}
assert train_groups.isdisjoint(dev_groups)
print(
    {
        "checks": "passed",
        "rows_by_mode": splits,
        "long_pairs": sum(item["length"] == "long" for item in support),
        "max_chat_tokens": max(
            len(row["input_ids"]) for split in prepared.values() for row in split
        ),
    }
)

The `validation` split above is internal development data for checkpoint selection. The original 68 accepted cases stay in the separate benchmark repository for the fixed before/after comparison. The targets here passed automatic checks and Sol review. They do not have independent human labels.

The next cell selects 16 training rows for the first L4 training proof. It does not train a model. These are training examples to memorize, so they cannot measure generalization.


In [ ]:
train_meta = [item for item in support if item["split"] == "train"]
selected = []
for mode in ("/slack", "/transcribe", "/prompt", "/polish"):
    for length in ("short", "long"):
        for variant in ("identity", "mode_noise"):
            index = next(
                i
                for i, item in enumerate(train_meta)
                if item["length"] == length
                and item["variant"] == variant
                and rows_by_split["train"][i]["user"].startswith(f"MODE: {mode}\n")
            )
            selected.append(index)
assert len(set(selected)) == 16
proof_path = Path("rewriter-memorization-16.jsonl")
proof_path.write_text(
    "".join(json.dumps(rows_by_split["train"][i], ensure_ascii=False) + "\n" for i in selected)
)
report = {
    "repo": repo,
    "revision": revision,
    "splits": quality["splits"],
    "file_hashes": manifest["files"],
    "assistant_only_loss_checked": True,
    "selected_training_ids": [train_meta[i]["id"] for i in selected],
    "student_training_run": False,
}
Path("rewriter-data-readiness.json").write_text(json.dumps(report, indent=2) + "\n")
print(
    {
        "proof_rows": len(selected),
        "proof_file": str(proof_path),
        "readiness_receipt": "rewriter-data-readiness.json",
        "training_run": False,
    }
)